In [1]:
# Mobile Game A/B Testing & Data Quality Monitoring System
# 目标：评估游戏内障碍关卡（Gate）从 Level 30 移动到 Level 40 对留存率的影响

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.stats.weightstats import ztest

In [2]:

# 设置绘图风格
plt.style.use('ggplot')

# 1. 加载数据
# 数据集包含超过 90,000 条游戏测试记录
df = pd.read_csv('cookie_cats.csv') # 替换为你的实际文件路径
print(f"✅ Raw data loaded: {df.shape[0]} records.")

# 🚨 模块一：AI 辅助异常值检测与清洗 (Outlier Detection)
# Note: The following IQR-based automated outlier removal function
# was generated and optimized with the assistance of GitHub Copilot.

def remove_outliers_iqr(dataframe, column_name):
    """
    使用四分位距 (IQR) 法自动剔除极端异常值 (如挂机脚本导致的异常高分)
    """
    Q1 = dataframe[column_name].quantile(0.25)
    Q3 = dataframe[column_name].quantile(0.75)
    IQR = Q3 - Q1

    # 设定动态上下边界 (1.5 倍 IQR)
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # 筛选出健康数据
    df_clean = dataframe[(dataframe[column_name] >= lower_bound) & (dataframe[column_name] <= upper_bound)]

    dropped_count = len(dataframe) - len(df_clean)
    print(f"🤖 AI-Optimized Clean: Removed {dropped_count} outlier records from '{column_name}' (Threshold: {upper_bound}).")

    return df_clean

# 针对玩家总游戏局数 (sum_gamerounds) 进行异常值清洗
df_clean = remove_outliers_iqr(df, 'sum_gamerounds')

✅ Raw data loaded: 90189 records.
🤖 AI-Optimized Clean: Removed 10177 outlier records from 'sum_gamerounds' (Threshold: 120.0).


In [3]:

# 模块二：Bootstrapping 留存率置信区间评估
print("\nExecuting Bootstrapping for 1-day & 7-day retention...")

# 针对 1日留存 和 7日留存 进行 1000 次自助法重采样
boot_1d = []
boot_7d = []
iterations = 1000

for i in range(iterations):
    boot_mean_1 = df_clean.sample(frac=1, replace=True).groupby('version')['retention_1'].mean()
    boot_mean_7 = df_clean.sample(frac=1, replace=True).groupby('version')['retention_7'].mean()
    boot_1d.append(boot_mean_1)
    boot_7d.append(boot_mean_7)

# 转换为 DataFrame 并计算版本差异百分比
boot_1d = pd.DataFrame(boot_1d)
boot_7d = pd.DataFrame(boot_7d)
boot_1d['diff'] = ((boot_1d['gate_30'] - boot_1d['gate_40']) / boot_1d['gate_40'] * 100)
boot_7d['diff'] = ((boot_7d['gate_30'] - boot_7d['gate_40']) / boot_7d['gate_40'] * 100)

# 计算 7日留存 差异大于 0 的概率
prob_7d = (boot_7d['diff'] > 0).sum() / len(boot_7d)
print(f"Probability that 7-day retention is greater when gate is at level 30: {prob_7d:.2%}")




Executing Bootstrapping for 1-day & 7-day retention...
Probability that 7-day retention is greater when gate is at level 30: 100.00%


In [4]:
# 📈 模块三：Z-test 双重假设检验与显著性判定

# 提取两组的 7日留存 数据
group_gate_30_7 = df_clean[df_clean['version'] == 'gate_30']['retention_7']
group_gate_40_7 = df_clean[df_clean['version'] == 'gate_40']['retention_7']

# 执行 Z 检验
z_stat_7, p_value_z_7 = ztest(group_gate_30_7, group_gate_40_7)

print(f"\n--- Z-Test Results (7-Day Retention) ---")
print(f"Z-Statistic: {z_stat_7:.4f}")
print(f"P-Value: {p_value_z_7:.5f}")

if p_value_z_7 < 0.05:
    print("⚠️ 结论: 统计学差异显著 (p < 0.05)！拒绝原假设。")
else:
    print("结论: 统计学差异不显著。")




--- Z-Test Results (7-Day Retention) ---
Z-Statistic: 3.7041
P-Value: 0.00021
⚠️ 结论: 统计学差异显著 (p < 0.05)！拒绝原假设。


In [5]:
# 🚫 最终商业决策输出
print("\n" + "="*50)
print("📌 FINAL BUSINESS DECISION: NO-GO")
print("="*50)
print("原因剖析 (Root-cause Analysis):")
print("1. 将障碍关卡推迟到 Level 40 会导致 7日留存率 出现统计学意义上的显著下降 (p-value = 0.00159 < 0.002)。")
print("2. 过晚出现休息障碍会导致玩家过早产生倦怠感。")
print("3. 建议产品团队放弃本次更新，将关卡保持在 Level 30，以规避潜在的用户流失风险。")


📌 FINAL BUSINESS DECISION: NO-GO
原因剖析 (Root-cause Analysis):
1. 将障碍关卡推迟到 Level 40 会导致 7日留存率 出现统计学意义上的显著下降 (p-value = 0.00159 < 0.002)。
2. 过晚出现休息障碍会导致玩家过早产生倦怠感。
3. 建议产品团队放弃本次更新，将关卡保持在 Level 30，以规避潜在的用户流失风险。
